# 🚀 Mars to Table × Operations Research
## MILP Optimization of a Food System for a Mars Mission

**Operations Research (OR) / Supply Chain Analytics case study**

This notebook transforms a food-sustainability problem for a long-duration mission into a **Mixed-Integer Linear Programming (MILP)** model using **PuLP + CBC**.

### Decision question

> **What combination of crops and alternative protein sources should a Mars crew operate to reduce dependence on Earth while simultaneously respecting area, water, biomass, nutrition, and diversity limits?**

The model also explores the **trade-off between self-sufficiency and diversity** through different values of `K`, the minimum number of active species.

### Official reference

NASA describes **Mars to Table** as a challenge to design food systems capable of supporting extended missions to the Moon and Mars, considering nutrition, safety, usability, integration with life-support systems, and system operation.  
**Official source:** [NASA — Mars to Table Challenge](https://www.nasa.gov/prizes-challenges-and-crowdsourcing/marstotable/)

> **Note:** this notebook is an academic/analytical case inspired by NASA's public challenge. It is not an official NASA model and does not imply NASA sponsorship or validation.


## 🎯 What this project demonstrates

This case is designed to demonstrate skills applicable to positions in:

- Operations Research / Optimization
- Supply Chain Analytics
- Decision Science
- Mathematical Optimization
- Network & Resource Planning
- Data Science applied to operations

### Techniques used

1. **MILP / Mixed-Integer Linear Programming**
2. Continuous production variables `x[i,t]`
3. Binary activation variables `y[i]`
4. Nutritional constraints
5. Capacity and resource constraints
6. Maturity/growth-cycle constraints
7. Diversity constraint
8. Earth-dependency penalty
9. Water footprint and water recovery
10. Sensitivity analysis
11. `K` scenario comparison
12. Export of results to Excel


In [2]:
# ============================================================
# 0. SETUP — Google Colab
# ============================================================

!pip -q install pulp openpyxl pandas matplotlib

import os

# Download the Excel file from GitHub
GITHUB_RAW_URL = 'https://raw.githubusercontent.com/LeoSanta15/mars-to-table-optimization/main/candidatos_mars_to_table.xlsx'
EXCEL_FILENAME = 'candidatos_mars_to_table.xlsx'
!wget -O {EXCEL_FILENAME} {GITHUB_RAW_URL}

EXCEL_PATH = EXCEL_FILENAME
print(f"Selected file: {EXCEL_PATH}")


--2026-08-16 22:51:23--  https://raw.githubusercontent.com/LeoSanta15/mars-to-table-optimization/main/candidatos_mars_to_table.xlsx
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 21019 (21K) [application/octet-stream]
Saving to: ‘candidatos_mars_to_table.xlsx’

candidatos_mars_to_ 100%[===================>]  20.53K  --.-KB/s    in 0.006s  

2026-08-16 22:51:24 (3.51 MB/s) - ‘candidatos_mars_to_table.xlsx’ saved [21019/21019]

Archivo seleccionado: candidatos_mars_to_table.xlsx


## 📦 Input data

The model uses an Excel workbook with two main sheets:

- `Cultivos_Fotosinteticos`
- `Proteinas_Alternativas`

The script validates the data before building the model and reports warnings or critical errors.

**Important:** the Excel file was not included in the Python file originally provided for this conversation. Therefore, the first cell downloads it directly into Colab.


### Model


In [6]:
"""
Mars to Table — Corrected MILP Model (PuLP), v3
================================================
Resolves the issues identified in v2:

1. [HIGH] Alternative proteins NOW contribute calories (r_kcal_kg calculado
   from literature data: FAO, USDA, studies on insects as food).

2. [ALTA] Duplicated Section 6b renamed -> 6d (net water with global rates).
   The original Section 6b (candidate-specific rates) is preserved.

3. [MEDIUM] Input-data validation: checks positivity, handles NaN
   with documented default values, and aborts with a clear message if
   critical data is missing.

4. [MEDIA] Objective-function weights parameterized (no more magic numbers).

5. [MEDIA] Improved documentation of variables and units.

6. [LOW] Additional charts: production-mix evolution for multiple K values,
   net-water sensitivity vs. recycling rate.

Reads 'candidatos_mars_to_table.xlsx' (13 photosynthetic crops + 5 alternative proteins
including Moringa oleifera as an arid-zone candidate).
"""

import math
import sys
import pandas as pd
import pulp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# EXCEL_PATH was already defined in the Colab loading cell.


### 0. CONFIGURABLE PARAMETERS (previously hard-coded magic numbers)


In [7]:
# ------------------------------------------------------------------
# 0. CONFIGURABLE PARAMETERS (previously hard-coded magic numbers)
WEIGHT_EARTH_DEPENDENCY = 1000.0   # weight of Earth dependency in objective
WEIGHT_WATER = 0.001               # weight of water in objective (tie-breaker)
BIG_M = 1e5                      # constante Big-M para activacion
MIN_AVG_KG_DIA = 0.05            # minimum average production for a species to "count"


### 1. MISSION SCENARIO


In [8]:
# ------------------------------------------------------------------
N_CREW = 4
T_WEEKS = 16
RDA_PROTEIN_G = 56
RDA_KCAL = 2500
A_MAX_M2 = 250
W_MAX_L_DIA = 2000
B_MAX_KG_DIA = 100
EARTH_CAP_START = 0.50
EARTH_CAP_END = 0.30
MIN_SPECIES_VALUES = [1, 3, 5, 7, 9, 11, 13, 15, 17]

d_protein = N_CREW * RDA_PROTEIN_G
d_kcal = N_CREW * RDA_KCAL


def earth_cap(t):
    """Maximum fraction of the deficit that can come from Earth in week t.
    Decreases linearly from EARTH_CAP_START to EARTH_CAP_END."""
    if T_WEEKS == 1:
        return EARTH_CAP_START
    frac = (t - 1) / (T_WEEKS - 1)
    return EARTH_CAP_START - (EARTH_CAP_START - EARTH_CAP_END) * frac


### 2. AUXILIARY DATA: calories from alternative proteins


In [9]:
# ------------------------------------------------------------------
# These values do NOT exist in the original Excel. They are calculated from
# typical nutritional composition reported in the literature:
#   - Grillo: ~65% proteina, ~20% grasa, ~5% carbohidratos -> ~4500 kcal/kg
#   - Tenebrio: ~50% proteina, ~30% grasa -> ~5500 kcal/kg
#   - Micoproteina (Quorn): ~45% proteina, bajo grasa -> ~1100 kcal/kg
#   - Hongo ostra (fresco): ~3% proteina, ~1% grasa, ~5% carbs -> ~330 kcal/kg
#   - Carne cultivada: ~20% proteina, ~20% grasa -> ~2500 kcal/kg
# Sources: FAO Nutritional Database, USDA FoodData Central, Oonincx et al. 2015
# (J. Insects Food Feed), Finke 2002 (complete nutrient composition of insects).
ALT_KCAL_PER_KG = {
    "Grillo (Acheta domesticus)": 4500,
    "Tenebrio / gusano de la harina (Tenebrio molitor)": 5500,
    "Micoproteina / Fusarium venenatum (tipo Quorn)": 1100,
    "Hongo ostra / Oyster mushroom (Pleurotus ostreatus)": 330,
    "Carne cultivada / Cultivated meat (celulas animales in vitro)": 2500,
}

# Default value for oyster mushroom water use (NaN in the original Excel).
# Estimated by analogy with mycoprotein (ambos son hongos, cultivo en
# sustrato/solucion nutritiva). Pleurotus literature reports
# 0.42-1.26 m3/kg en base seca; because it is consumed fresh (~90% agua),
# el valor por kg fresco es lower. We use 1500 L/kg as a conservative estimate
# conservadora until experimental data is available.
DEFAULT_WATER_OYSTER_MUSHROOM = 1500.0


### 3. DATA LOADING WITH VALIDATION


In [14]:
# ------------------------------------------------------------------
def load_and_validate_data(excel_path):
    """Loads data from Excel and validates consistency. Aborts with a clear message
    if critical data is missing or invalid."""

    df1 = pd.read_excel(excel_path, sheet_name="Cultivos_Fotosinteticos", header=3).dropna(subset=["Candidato"])
    df2 = pd.read_excel(excel_path, sheet_name="Proteinas_Alternativas", header=3).dropna(subset=["Candidato"])

    crops, alt = {}, {}
    errors = []
    warnings = []

    # --- Validate photosynthetic crops ---
    required_cols_crops = {
        "r_proteina_g_m2_d": float,
        "r_calorico_kcal_m2_d": float,
        "a_i_m2_por_kg_comestible_dia": float,
        "w_i": "Agua_L_m2_d",
        "edible_g_m2d": "Biomasa_comestible_gDM_m2_d",
        "ciclo_dias": "Ciclo_dias",
    }

    for _, row in df1.iterrows():
        name = row["Candidato"]

        # Verify critical numeric fields
        r_prot = float(row["r_proteina_g_m2_d"])
        r_kcal = float(row["r_calorico_kcal_m2_d"])
        a_i = float(row["a_i_m2_por_kg_comestible_dia"])
        w_i = float(row["Agua_L_m2_d"])
        edible = float(row["Biomasa_comestible_gDM_m2_d"])
        ciclo = float(row["Ciclo_dias"])

        for label, val in [("r_proteina", r_prot), ("r_calorico", r_kcal),
                           ("a_i", a_i), ("w_i", w_i), ("edible", edible),
                           ("ciclo_dias", ciclo)]:
            if math.isnan(val) or val is None:
                errors.append(f"[{name}] {label} = NaN/None (critical data missing)")
            elif val < 0:
                errors.append(f"[{name}] {label} = {val} (negative, must be >= 0)")
            elif val == 0 and label in ("r_proteina", "r_calorico", "edible"):
                warnings.append(f"[{name}] {label} = 0 (crop does not contribute this nutrient)")

        if ciclo <= 0:
            errors.append(f"[{name}] ciclo_dias = {ciclo} (must be > 0)")

        # Verify consistency: calculated protein concentration vs. % DM
        if edible > 0:
            prot_conc_calc = r_prot / (edible / 1000)  # g proteina / kg biomasa
            prot_pct_dm = row.get("Proteina_pct_DM", float("nan"))
            if not math.isnan(prot_pct_dm):
                prot_conc_expected = float(prot_pct_dm) * 10  # g/kg
                if abs(prot_conc_calc - prot_conc_expected) > prot_conc_expected * 0.5:
                    warnings.append(
                        f"[{name}] Protein inconsistency: calculated={prot_conc_calc:.1f} g/kg "
                        f"vs. Proteina_pct_DM*10={prot_conc_expected:.1f} g/kg"
                    )

        crops[name] = {
            "tipo": "fotosintetico",
            "r_prot_m2d": r_prot,
            "r_kcal_m2d": r_kcal,
            "a_i": a_i,
            "w_i": w_i,
            "edible_g_m2d": edible,
            "ciclo_dias": ciclo,
        }

    # --- Validate alternative proteins ---
    for _, row in df2.iterrows():
        name = row["Candidato"]

        r_prot_kg = float(row["r_proteina_g_por_kg_producto"])
        agua = row["Agua_L_por_kg_producto"]
        ciclo = float(row["Ciclo_dias"])

        # Handle NaN in water use (oyster mushroom)
        if pd.isna(agua):
            agua = DEFAULT_WATER_OYSTER_MUSHROOM
            warnings.append(
                f"[{name}] Agua_L_por_kg_producto = NaN en Excel. "
                f"Using default value {DEFAULT_WATER_OYSTER_MUSHROOM} L/kg "
                f"(estimated by analogy with mycoprotein; validate experimentally)."
            )
        else:
            agua = float(agua)

        # Verify critical fields
        for label, val in [("r_proteina_g_por_kg", r_prot_kg), ("ciclo_dias", ciclo)]:
            if math.isnan(val) or val is None:
                errors.append(f"[{name}] {label} = NaN/None (critical data missing)")
            elif val < 0:
                errors.append(f"[{name}] {label} = {val} (negativo)")

        if ciclo <= 0:
            errors.append(f"[{name}] ciclo_dias = {ciclo} (must be > 0)")

        # Verify that calories are available for this candidate
        if name not in ALT_KCAL_PER_KG:
            errors.append(
                f"[{name}] No entry exists in ALT_KCAL_PER_KG. "
                f"Add estimated calories (kcal/kg product) to the dictionary."
            )
            r_kcal_kg = 0.0
        else:
            r_kcal_kg = ALT_KCAL_PER_KG[name]

        # Verify consistency: r_proteina_g_por_kg vs. Proteina_pct_producto
        prot_pct = row.get("Proteina_pct_producto", float("nan"))
        if not math.isnan(prot_pct):
            expected_prot = float(prot_pct) * 10
            if abs(r_prot_kg - expected_prot) > 1.0:
                warnings.append(
                    f"[{name}] Inconsistency: r_proteina_g_por_kg={r_prot_kg} "
                    f"vs. Proteina_pct_producto*10={expected_prot}"
                )

        alt[name] = {
            "tipo": "alternativo",
            "r_prot_kg": r_prot_kg,
            "r_kcal_kg": r_kcal_kg,  # <-- NEW: calories per kg of product
            "w_i": agua,
            "ciclo_dias": ciclo,
        }

    # Report and abort if there are critical errors
    if warnings:
        print("\n=== WARNINGS (non-blocking) ===")
        for w in warnings:
            print(f"  ⚠️  {w}")

    if errors:
        print("\n=== CRITICAL ERRORS (blocking) ===")
        for e in errors:
            print(f"  ❌  {e}")
        print("\n>>> Aborting: fix the errors in the Excel or code before continuing.")
        sys.exit(1)

    print(f"\n✅ Validation successful: {len(crops)} photosynthetic crops and {len(alt)} alternative proteins.")
    return crops, alt


crops, alt = load_and_validate_data(EXCEL_PATH)
all_ids = list(crops.keys()) + list(alt.keys())
weeks = list(range(1, T_WEEKS + 1))
print(f"Total candidates: {len(all_ids)}.")


def first_week(cycle_days):
    """First week in which the candidate can produce (after maturation)."""
    return math.ceil(cycle_days / 7)



=== ADVERTENCIAS (no bloqueantes) ===
  ⚠️  [Hongo ostra / Oyster mushroom (Pleurotus ostreatus)] Agua_L_por_kg_producto = NaN en Excel. Usando valor por defecto 1500.0 L/kg (estimado por analogia con micoproteina, validar experimentalmente).

✅ Validacion exitosa: 13 cultivos fotosinteticos y 5 proteinas alternativas.
Candidatos totales: 18.


### 4. FUNCTION THAT BUILDS AND SOLVES THE MODEL FOR A GIVEN K


In [15]:
# ------------------------------------------------------------------
def solve_model(min_species):
    """Solves the MILP model for a given value of K (minimum species).

    Variables:
      x[(i,t)]  : production of candidate i in week t (kg/dia de biomasa
                  comestible para fotosinteticos; kg/dia de producto para
                  alternative candidates)
      y[i]      : 1 1 if candidate i is active, 0 otherwise
      delta_prot[t] : protein deficit covered by Earth in week t (g/day)
      delta_kcal[t] : calorie deficit covered by Earth in week t (kcal/day)

    The objective function minimizes Earth dependency (dominant) and
    water consumption (secondary, tie-breaker).
    """
    prob = pulp.LpProblem(f"Mars_to_Table_K{min_species}", pulp.LpMinimize)

    x = {(i, t): pulp.LpVariable(f"x_{i}_{t}_{min_species}", lowBound=0)
         for i in all_ids for t in weeks}
    y = {i: pulp.LpVariable(f"y_{i}_{min_species}", cat="Binary")
         for i in all_ids}
    delta_prot = {t: pulp.LpVariable(f"dp_{t}_{min_species}", lowBound=0)
                  for t in weeks}
    delta_kcal = {t: pulp.LpVariable(f"dk_{t}_{min_species}", lowBound=0)
                  for t in weeks}

    # --- Water terms ---
    water_terms = (
        [crops[i]["w_i"] * crops[i]["a_i"] * x[(i, t)]
         for i in crops for t in weeks]
        + [alt[i]["w_i"] * x[(i, t)]
           for i in alt for t in weeks]
    )

    # --- Earth dependency ---
    earth_dependency = (
        pulp.lpSum(delta_prot[t] for t in weeks) / d_protein
        + pulp.lpSum(delta_kcal[t] for t in weeks) / d_kcal
    )

    # --- Objective function ---
    prob += (WEIGHT_EARTH_DEPENDENCY * earth_dependency
             + WEIGHT_WATER * pulp.lpSum(water_terms)), "Objetivo"

    # --- Weekly constraints ---
    for t in weeks:
        # Protein supply (g/day)
        prot_supply = (
            pulp.lpSum(
                (crops[i]["r_prot_m2d"] / (crops[i]["edible_g_m2d"] / 1000))
                * x[(i, t)]
                for i in crops
            )
            + pulp.lpSum(alt[i]["r_prot_kg"] * x[(i, t)] for i in alt)
        )
        prob += prot_supply + delta_prot[t] >= d_protein
        prob += delta_prot[t] <= earth_cap(t) * d_protein

        # Calorie supply (kcal/day)  <-- CORREGIDO: ahora incluye alternative candidates
        kcal_supply = (
            pulp.lpSum(
                (crops[i]["r_kcal_m2d"] / (crops[i]["edible_g_m2d"] / 1000))
                * x[(i, t)]
                for i in crops
            )
            + pulp.lpSum(alt[i]["r_kcal_kg"] * x[(i, t)] for i in alt)  # NUEVO
        )
        prob += kcal_supply + delta_kcal[t] >= d_kcal
        prob += delta_kcal[t] <= earth_cap(t) * d_kcal

        # Recursos
        prob += pulp.lpSum(crops[i]["a_i"] * x[(i, t)] for i in crops) <= A_MAX_M2
        prob += pulp.lpSum(x[(i, t)] for i in alt) <= B_MAX_KG_DIA
        prob += (
            pulp.lpSum(crops[i]["w_i"] * crops[i]["a_i"] * x[(i, t)]
                       for i in crops)
            + pulp.lpSum(alt[i]["w_i"] * x[(i, t)] for i in alt)
        ) <= W_MAX_L_DIA

        # Growth cycle and activation
        for i in all_ids:
            src = crops[i] if i in crops else alt[i]
            fw = first_week(src["ciclo_dias"])
            if t < fw:
                prob += x[(i, t)] == 0
            prob += x[(i, t)] <= BIG_M * y[i]

    # --- Diversity constraint: minimum production per active species ---
    for i in all_ids:
        src = crops[i] if i in crops else alt[i]
        fw = first_week(src["ciclo_dias"])
        active_weeks = [t for t in weeks if t >= fw]
        if active_weeks:
            prob += (pulp.lpSum(x[(i, t)] for t in active_weeks)
                     >= MIN_AVG_KG_DIA * len(active_weeks) * y[i])

    # --- Diversity constraint: minimum number of active species ---
    prob += pulp.lpSum(y[i] for i in all_ids) >= min_species, "Diversidad_minima"

    # --- Solve ---
    status = prob.solve(pulp.PULP_CBC_CMD(msg=0))

    result = {
        "K": min_species,
        "status": pulp.LpStatus[status],
        "model": prob,
        "vars": {"x": x, "y": y, "delta_prot": delta_prot, "delta_kcal": delta_kcal},
    }

    if pulp.LpStatus[status] == "Optimal":
        result["earth_dependency"] = pulp.value(earth_dependency)
        result["agua_total"] = sum(pulp.value(term) for term in water_terms)
        active = [i for i in all_ids if (y[i].value() or 0) > 0.5]
        result["especies_activas"] = active
        result["n_especies"] = len(active)
        result["weekly"] = {(i, t): (x[(i, t)].value() or 0)
                            for i in all_ids for t in weeks}
        # NEW: additional metrics
        result["area_usada_prom"] = sum(
            pulp.lpSum(crops[i]["a_i"] * x[(i, t)] for i in crops).value()
            for t in weeks
        ) / T_WEEKS
        result["biomasa_alt_prom"] = sum(
            pulp.lpSum(x[(i, t)] for i in alt).value()
            for t in weeks
        ) / T_WEEKS
    else:
        result["earth_dependency"] = None
        result["agua_total"] = None
        result["especies_activas"] = []
        result["n_especies"] = 0
        result["weekly"] = {}
        result["area_usada_prom"] = None
        result["biomasa_alt_prom"] = None

    return result


### 5. RUN THE MODEL FOR EACH K VALUE


In [16]:
# ------------------------------------------------------------------
resultados = [solve_model(k) for k in MIN_SPECIES_VALUES]

print("\n=== Comparacion: costo de forzar diversidad de especies ===")
comp_rows = []
for r in resultados:
    comp_rows.append({
        "K_especies_minimas": r["K"],
        "Estado": r["status"],
        "Especies_activas": r["n_especies"],
        "Dependencia_Tierra": round(r["earth_dependency"], 4)
        if r["earth_dependency"] is not None else None,
        "Agua_total_prom_L_dia": round(r["agua_total"] / T_WEEKS, 1)
        if r["agua_total"] is not None else None,
        "Area_usada_prom_m2": round(r["area_usada_prom"], 1)
        if r["area_usada_prom"] is not None else None,
        "Biomasa_alt_prom_kg_dia": round(r["biomasa_alt_prom"], 2)
        if r["biomasa_alt_prom"] is not None else None,
        "Lista_especies": ", ".join(r["especies_activas"])
        if r["especies_activas"] else "-",
    })
comp_df = pd.DataFrame(comp_rows)
print(comp_df.drop(columns=["Lista_especies"]).to_string(index=False))
print()
for r in resultados:
    print(f"K={r['K']:>2} ({r['status']}): "
          f"{', '.join(r['especies_activas']) if r['especies_activas'] else '(infactible)'}")



=== Comparacion: costo de forzar diversidad de especies ===
 K_especies_minimas  Estado  Especies_activas  Dependencia_Tierra  Agua_total_prom_L_dia  Area_usada_prom_m2  Biomasa_alt_prom_kg_dia
                  1 Optimal                 5                 0.0                 1066.8               147.9                     0.00
                  3 Optimal                 5                 0.0                 1066.8               147.9                     0.00
                  5 Optimal                 5                 0.0                 1066.8               147.9                     0.00
                  7 Optimal                 7                 0.0                 1069.8               151.1                     0.00
                  9 Optimal                 9                 0.0                 1080.0               157.4                     0.00
                 11 Optimal                11                 0.0                 1093.1               157.3                     0.04
 

### 6a. NET MAKE-UP WATER WITH CANDIDATE-SPECIFIC RECYCLING RATES


In [17]:
# ------------------------------------------------------------------
# Key physical principle for a SEALED habitat: the water that evaporates
# from photosynthetic crops is NOT lost to the external atmosphere -- it remains
# as humidity inside the habitat and is recovered by the HVAC. Alternative proteins
# alternativas have lower rates because some of the water remains embedded
# in solid residues (sustrato de hongo, frass de insectos, medio celular).
#
# IMPORTANT: these rates are REASONED ESTIMATES based on
# de ECLSS/raceways -- there is no study that measures "tasa de reciclaje de
# agua por candidato de sistema alimentario espacial". Treat them as
# engineering assumptions to be validated, not hard data.
RECYCLE_RATE_BY_CANDIDATE = {
    # --- Hydroponic photosynthetic crops ---
    "Trigo (Triticum aestivum)": 0.96,
    "Soya (Glycine max)": 0.96,
    "Papa (Solanum tuberosum)": 0.96,
    "Camote (Ipomoea batatas)": 0.96,
    "Lechuga (Lactuca sativa)": 0.96,
    "Tomate (Solanum lycopersicum)": 0.96,
    "Arroz (Oryza sativa)": 0.95,
    "Mani / Peanut (Arachis hypogaea)": 0.95,
    "Quinua (Chenopodium quinoa)": 0.95,
    "Moringa (Moringa oleifera)": 0.95,
    # --- Microalgae/aquatic macrophytes ---
    "Espirulina (Arthrospira platensis)": 0.94,
    "Chlorella (Chlorella vulgaris)": 0.94,
    "Lenteja de agua / Duckweed (Lemna minor)": 0.94,
    # --- Alternative proteins ---
    "Grillo (Acheta domesticus)": 0.75,
    "Tenebrio / gusano de la harina (Tenebrio molitor)": 0.75,
    "Micoproteina / Fusarium venenatum (tipo Quorn)": 0.80,
    "Hongo ostra / Oyster mushroom (Pleurotus ostreatus)": 0.65,
    "Carne cultivada / Cultivated meat (celulas animales in vitro)": 0.70,
}
DEFAULT_RECYCLE_RATE = 0.90

print("\n=== 6a. NET water de make-up con candidate-specific recycling rate (K=7) ===")
k7_result = next(r for r in resultados if r["K"] == 7)
per_candidate_rows = []
total_bruta = 0.0
total_neta = 0.0
for i in all_ids:
    bruta_i = 0.0
    for t in weeks:
        v = k7_result["weekly"].get((i, t), 0)
        if i in crops:
            bruta_i += crops[i]["w_i"] * crops[i]["a_i"] * v
        else:
            bruta_i += alt[i]["w_i"] * v
    if bruta_i <= 1e-9:
        continue
    rate = RECYCLE_RATE_BY_CANDIDATE.get(i, DEFAULT_RECYCLE_RATE)
    neta_i = bruta_i * (1 - rate)
    total_bruta += bruta_i
    total_neta += neta_i
    per_candidate_rows.append({
        "Candidato": i,
        "Agua_bruta_16sem_L": round(bruta_i * 7, 1),
        "Tasa_reciclaje_pct": round(rate * 100, 1),
        "Agua_neta_16sem_L": round(neta_i * 7, 1),
    })

per_df = pd.DataFrame(per_candidate_rows).sort_values("Agua_bruta_16sem_L", ascending=False)
print(per_df.to_string(index=False))
tasa_efectiva = 1 - (total_neta / total_bruta) if total_bruta else 0
print(f"\nTotal agua bruta (16 sem): {total_bruta*7:,.0f} L (~{total_bruta*7/1000:,.2f} m3)")
print(f"Total agua NETA de make-up (16 sem, tasas diferenciadas): {total_neta*7:,.0f} L (~{total_neta*7/1000:,.2f} m3)")
print(f"Tasa de reciclaje EFECTIVA del sistema completo (ponderada): {tasa_efectiva*100:.1f}%")



=== 6a. Agua NETA de reposicion con tasa de reciclaje diferenciada por candidato (K=7) ===
                     Candidato  Agua_bruta_16sem_L  Tasa_reciclaje_pct  Agua_neta_16sem_L
Chlorella (Chlorella vulgaris)            104815.8                94.0             6288.9
      Papa (Solanum tuberosum)             13043.5                96.0              521.7
    Moringa (Moringa oleifera)              1228.7                95.0               61.4
     Trigo (Triticum aestivum)               727.9                96.0               29.1

Total agua bruta (16 sem): 119,816 L (~119.82 m3)
Total agua NETA de reposicion (16 sem, tasas diferenciadas): 6,901 L (~6.90 m3)
Tasa de reciclaje EFECTIVA del sistema completo (ponderada): 94.2%


### 6b. COMPLETE WATER BALANCE: transpiration vs. water embedded in harvested biomass


In [18]:
# ------------------------------------------------------------------
# w_i is the TOTAL water delivered to the hydroponic system (riego + solucion
# nutritiva + acido de pH) segun metodologia NASA (Wheeler et al. 1999).
# It is separated into:
#   1) TRANSPIRATED water: se condensa en HVAC y se recicla -> tasa RECYCLE_RATE.
#   2) EMBEDDED water en biomasa COSECHADA: sale con el alimento, se recupera
#      via ECLSS humano (ISS 2023: 98% recuperacion).
# Los valores de Biomasa_comestible_gDM_m2_d estan en DRY basis (NASA, 70C/72h).
DM_FRACTION = {
    "Trigo (Triticum aestivum)": 0.88,
    "Soya (Glycine max)": 0.90,
    "Papa (Solanum tuberosum)": 0.20,
    "Camote (Ipomoea batatas)": 0.23,
    "Lechuga (Lactuca sativa)": 0.05,
    "Tomate (Solanum lycopersicum)": 0.06,
    "Arroz (Oryza sativa)": 0.88,
    "Mani / Peanut (Arachis hypogaea)": 0.92,
    "Quinua (Chenopodium quinoa)": 0.88,
    "Moringa (Moringa oleifera)": 0.90,
    "Espirulina (Arthrospira platensis)": 0.92,
    "Chlorella (Chlorella vulgaris)": 0.92,
    "Lenteja de agua / Duckweed (Lemna minor)": 0.10,
}
ECLSS_HUMANO_RECOVERY = 0.98

print("\n=== 6b. Complete water balance K=7: transpiration vs. embedded water ===")
balance_rows = []
total_transp_bruta = total_transp_neta = 0.0
total_embed_bruta = total_embed_neta = 0.0
for i in all_ids:
    if i not in crops:
        continue
    dry_kg_16sem = sum(k7_result["weekly"].get((i, t), 0) for t in weeks) * 7
    if dry_kg_16sem <= 1e-9:
        continue
    dm = DM_FRACTION.get(i, 0.90)
    fresh_kg_16sem = dry_kg_16sem / dm
    embebida_L = fresh_kg_16sem - dry_kg_16sem
    bruta_total_i = next(
        row["Agua_bruta_16sem_L"] for row in per_candidate_rows if row["Candidato"] == i
    )
    transp_L = max(bruta_total_i - embebida_L, 0)

    rate_transp = RECYCLE_RATE_BY_CANDIDATE.get(i, DEFAULT_RECYCLE_RATE)
    transp_neta = transp_L * (1 - rate_transp)
    embebida_neta = embebida_L * (1 - ECLSS_HUMANO_RECOVERY)

    total_transp_bruta += transp_L
    total_transp_neta += transp_neta
    total_embed_bruta += embebida_L
    total_embed_neta += embebida_neta

    balance_rows.append({
        "Candidato": i,
        "Agua_transpirada_16sem_L": round(transp_L, 1),
        "Agua_embebida_en_cosecha_16sem_L": round(embebida_L, 1),
        "pct_embebida_del_total": round(100 * embebida_L / bruta_total_i, 2)
        if bruta_total_i else 0,
        "Neta_transpiracion_L": round(transp_neta, 2),
        "Neta_embebida_L": round(embebida_neta, 2),
    })

balance_df = pd.DataFrame(balance_rows)
print(balance_df.to_string(index=False))

total_bruta_16 = total_transp_bruta + total_embed_bruta
total_neta_16 = total_transp_neta + total_embed_neta
print(f"\nTranspired water (recirculated in habitat): "
      f"{total_transp_bruta:,.0f} L bruta -> {total_transp_neta:,.1f} L neta")
print(f"Water embedded in harvest (recovered through crew ECLSS): "
      f"{total_embed_bruta:,.1f} L bruta -> {total_embed_neta:,.2f} L neta")
print(f"TOTAL agua neta de make-up (ambas rutas): {total_neta_16:,.1f} L "
      f"(~{total_neta_16/1000:,.3f} m3) en 16 semanas")



=== 6b. Balance hidrico completo K=7: transpiracion vs. embebida ===
                     Candidato  Agua_transpirada_16sem_L  Agua_embebida_en_cosecha_16sem_L  pct_embebida_del_total  Neta_transpiracion_L  Neta_embebida_L
     Trigo (Triticum aestivum)                     727.7                               0.2                    0.03                 29.11             0.00
      Papa (Solanum tuberosum)                   12803.5                             240.0                    1.84                512.14             4.80
Chlorella (Chlorella vulgaris)                  104795.3                              20.5                    0.02               6287.72             0.41
    Moringa (Moringa oleifera)                    1228.3                               0.4                    0.03                 61.41             0.01

Agua transpirada (recirculada en habitat): 119,555 L bruta -> 6,890.4 L neta
Agua embebida en cosecha (recirculada via ECLSS tripulacion): 261.2 L bruta -> 5.2

### 6c. NET MAKE-UP WATER: sensitivity to global recycling rates


In [19]:
# ------------------------------------------------------------------
# Simplified approach: one global recycling rate for the entire system.
# Useful for sensitivity analysis and comparison with BLSS literature.
WATER_RECYCLE_RATES = [0.90, 0.93, 0.95, 0.98]

print("\n=== 6c. NET water de make-up by GLOBAL system recovery rate ===")
net_rows = []
for r in resultados:
    if r["status"] != "Optimal":
        continue
    # r["agua_total"] = suma sobre T_WEEKS semanas del agua diaria (L/dia * 1 semana)
    # = promedio diario * T_WEEKS (porque cada semana suma 7 * promedio_diario_semana)
    # Pero en realidad es: para cada semana t, sumamos agua_dia_t * 1 (la unidad es semana)
    # Entonces agua_total / T_WEEKS = promedio diario sobre toda la mision
    agua_bruta_dia = r["agua_total"] / T_WEEKS
    agua_bruta_total_16sem = agua_bruta_dia * T_WEEKS * 7  # 7 dias/semana
    row = {
        "K": r["K"],
        "Agua_bruta_L_dia": round(agua_bruta_dia, 1),
        "Agua_bruta_total_16sem_L": round(agua_bruta_total_16sem, 0),
    }
    for rate in WATER_RECYCLE_RATES:
        row[f"Neta_{int(rate*100)}pct_L_dia"] = round(agua_bruta_dia * (1 - rate), 2)
        row[f"Neta_{int(rate*100)}pct_total_16sem_L"] = round(
            agua_bruta_total_16sem * (1 - rate), 1
        )
    net_rows.append(row)

net_df = pd.DataFrame(net_rows)
cols_display = ["K", "Agua_bruta_L_dia"] + [f"Neta_{int(r*100)}pct_L_dia"
                                            for r in WATER_RECYCLE_RATES]
print(net_df[cols_display].to_string(index=False))

k7 = next((r for r in net_rows if r["K"] == 7), None)
if k7:
    print(f"\n--- Detalle K=7 (recommended scenario), 16 semanas ---")
    print(f"Gross water (throughput): {k7['Agua_bruta_total_16sem_L']:,.0f} L "
          f"(~{k7['Agua_bruta_total_16sem_L']/1000:,.1f} m3)")
    for rate in WATER_RECYCLE_RATES:
        total_neta = k7[f"Neta_{int(rate*100)}pct_total_16sem_L"]
        dia_neta = k7[f"Neta_{int(rate*100)}pct_L_dia"]
        print(f"  Con {int(rate*100)}% reciclaje -> make-up neta: "
              f"{dia_neta:,.2f} L/dia | {total_neta:,.0f} L total "
              f"({total_neta/1000:,.2f} m3)")



=== 6c. Agua NETA de reposicion segun tasa de recuperacion GLOBAL del sistema ===
 K  Agua_bruta_L_dia  Neta_90pct_L_dia  Neta_93pct_L_dia  Neta_95pct_L_dia  Neta_98pct_L_dia
 1            1066.8            106.68             74.67             53.34             21.34
 3            1066.8            106.68             74.67             53.34             21.34
 5            1066.8            106.68             74.67             53.34             21.34
 7            1069.8            106.98             74.88             53.49             21.40
 9            1080.0            108.00             75.60             54.00             21.60
11            1093.1            109.31             76.52             54.66             21.86
13            1110.2            111.02             77.71             55.51             22.20
15            1136.1            113.61             79.52             56.80             22.72
17            1241.4            124.14             86.90             62.07      

### 6d. COMPARISON: candidate-specific rates vs. a uniform global rate


In [20]:
# ------------------------------------------------------------------
# NEW: compara el resultado del complete water balance (6b) con el
# simplified assumption de tasa global uniforme.
print("\n=== 6d. Comparison: complete water balance vs. uniform global rate ===")
if k7:
    # Agua neta del balance completo (seccion 6b)
    neta_balance_completo = total_neta_16
    # Agua neta con tasa global uniforme de 95% (supuesto original v1)
    agua_bruta_total = k7["Agua_bruta_total_16sem_L"]
    neta_global_95 = agua_bruta_total * (1 - 0.95)
    neta_global_90 = agua_bruta_total * (1 - 0.90)
    neta_global_98 = agua_bruta_total * (1 - 0.98)

    print(f"Gross water total (K=7): {agua_bruta_total:,.0f} L")
    print(f"  -> COMPLETE water balance (6b): {neta_balance_completo:,.1f} L neta")
    print(f"  -> Uniform global assumption 95% uniforme:  {neta_global_95:,.0f} L neta")
    print(f"  -> Uniform global assumption 90% uniforme:  {neta_global_90:,.0f} L neta")
    print(f"  -> Uniform global assumption 98% uniforme:  {neta_global_98:,.0f} L neta")

    tasa_global_equivalente = 1 - (neta_balance_completo / agua_bruta_total)
    print(f"\nGLOBAL recycling rate equivalent to the complete balance: "
          f"{tasa_global_equivalente*100:.1f}%")
    print(f"(The optimal mix K=7 tiene una tasa efectiva de {tasa_global_equivalente*100:.1f}%, "
          f"which is {'higher' if tasa_global_equivalente > 0.95 else 'lower'} than the assumption "
          f"of a simplified uniform 95%)")



=== 6d. Comparacion: balance hidrico completo vs. tasa global uniforme ===
Agua bruta total (K=7): 119,816 L
  -> Balance hidrico COMPLETO (6b): 6,895.6 L neta
  -> Supuesto global 95% uniforme:  5,991 L neta
  -> Supuesto global 90% uniforme:  11,982 L neta
  -> Supuesto global 98% uniforme:  2,396 L neta

Tasa de reciclaje GLOBAL equivalente al balance completo: 94.2%
(La mezcla optima K=7 tiene una tasa efectiva de 94.2%, que es menor que el supuesto simplificado de 95% uniforme)


### 7. EXPORT RESULTS TO EXCEL


In [21]:
# ------------------------------------------------------------------
with pd.ExcelWriter("resultados_diversidad_v3.xlsx", engine="openpyxl") as writer:
    # Sheet 1: comparacion de K
    comp_df.to_excel(writer, sheet_name="Comparacion_K", index=False)

    # Sheet 2-9: weekly detail para cada K factible
    for r in resultados:
        if r["status"] != "Optimal":
            continue
        rows = [{"Candidato": i, "Semana": t, "Produccion_kg_dia": v}
                for (i, t), v in r["weekly"].items() if v > 1e-6]
        if rows:
            pd.DataFrame(rows).to_excel(writer, sheet_name=f"Detalle_K{r['K']}", index=False)

    # Sheet 10: net water by candidate (tasas diferenciadas)
    per_df.to_excel(writer, sheet_name="Agua_neta_por_candidato_K7", index=False)

    # Sheet 11: complete water balance
    balance_df.to_excel(writer, sheet_name="Balance_hidrico_completo_K7", index=False)

    # Sheet 12: net water with global rates
    net_df.to_excel(writer, sheet_name="Agua_neta_make-up_global", index=False)

print("\n✅ Results exported to resultados_diversidad_v3.xlsx")



✅ Resultados exportados a resultados_diversidad_v3.xlsx


### 8. CHARTS


In [22]:
# ------------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

factibles = [r for r in resultados if r["status"] == "Optimal"]
ks = [r["K"] for r in factibles]
dep = [r["earth_dependency"] for r in factibles]
agua = [r["agua_total"] / T_WEEKS for r in factibles]

# --- Grafico 1: Diversity vs. cost trade-off ---
ax1 = axes[0, 0]
ax1.plot(ks, dep, marker="o", color="firebrick", linewidth=2,
         label="Earth dependency (indice)")
ax1.set_xlabel("K = minimum number of active species")
ax1.set_ylabel("Earth dependency (indice, 0=autosuficiente)", color="firebrick")
ax1.set_title("Cost of diversity: self-sufficiency")
ax1.tick_params(axis="y", labelcolor="firebrick")
ax1.grid(alpha=0.3)

ax1b = ax1.twinx()
ax1b.plot(ks, agua, marker="s", color="steelblue", linewidth=2,
          label="Total water (L/day avg.)")
ax1b.set_ylabel("Total water (average L/day)", color="steelblue")
ax1b.tick_params(axis="y", labelcolor="steelblue")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", fontsize=8)

# --- Grafico 2: Production mix para K=7 ---
ax2 = axes[0, 1]
k7 = next((r for r in factibles if r["K"] == 7), None)
if k7:
    weekly_df = pd.DataFrame(
        [{"Candidato": i, "Semana": t, "Produccion_kg_dia": v}
         for (i, t), v in k7["weekly"].items() if v > 1e-6]
    )
    pivot = weekly_df.pivot(index="Semana", columns="Candidato",
                            values="Produccion_kg_dia").fillna(0)
    pivot = pivot.reindex(weeks, fill_value=0)
    pivot.plot.area(ax=ax2, alpha=0.85, linewidth=0)
    ax2.set_title(f"Production mix — K=7 minimum species")
    ax2.set_xlabel("Mission week")
    ax2.set_ylabel("Production (kg/day)")
    ax2.legend(loc="upper left", fontsize=6, ncol=2)
    ax2.grid(alpha=0.3)

# --- Grafico 3: Production-mix evolution for multiple K ---
ax3 = axes[1, 0]
for r in factibles:
    if r["K"] in [1, 5, 7, 11]:
        weekly_total = [sum(r["weekly"].get((i, t), 0) for i in all_ids)
                        for t in weeks]
        ax3.plot(weeks, weekly_total, marker="o", label=f"K={r['K']}", alpha=0.7)
ax3.set_xlabel("Mission week")
ax3.set_ylabel("Total production (kg/day)")
ax3.set_title("Total production evolution by K")
ax3.legend(loc="best", fontsize=8)
ax3.grid(alpha=0.3)

# --- Grafico 4: Net-water sensitivity vs. recycling rate ---
ax4 = axes[1, 1]
if k7:
    rates = [0.85, 0.90, 0.93, 0.95, 0.98, 0.99]
    agua_bruta = k7["agua_total"] / T_WEEKS * T_WEEKS * 7  # total 16 sem
    netas = [agua_bruta * (1 - r) for r in rates]
    ax4.plot([r * 100 for r in rates], [n / 1000 for n in netas],
             marker="o", color="darkgreen", linewidth=2)
    ax4.axvline(x=95, color="gray", linestyle="--", alpha=0.5, label="Original assumption (95%)")
    ax4.set_xlabel("Global recycling rate (%)")
    ax4.set_ylabel("Agua neta de make-up (m3 en 16 semanas)")
    ax4.set_title("Sensitivity: net water vs. recycling rate (K=7)")
    ax4.legend(fontsize=8)
    ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("diversidad_tradeoff_v3.png", dpi=150)
print("Chart saved to diversidad_tradeoff_v3.png")

print("\n" + "=" * 60)
print("SUMMARY OF CORRECTIONS APPLIED IN v3")
print("=" * 60)
print("""
1. [ALTA] Alternative proteins now contribute calories:
   - Grillo: 4500 kcal/kg
   - Tenebrio: 5500 kcal/kg
   - Micoproteina: 1100 kcal/kg
   - Hongo ostra: 330 kcal/kg
   - Carne cultivada: 2500 kcal/kg
   Fuente: FAO, USDA, Oonincx et al. 2015, Finke 2002

2. [ALTA] Duplicated Section 6b renamed:
   - 6a: tasas diferenciadas por candidato (preservada)
   - 6b: complete water balance (transpiration vs. embedded water)
   - 6c: tasas globales [90,93,95,98]% (antes era 6b duplicada)
   - 6d: comparacion balance completo vs. global (NUEVO)

3. [MEDIUM] Input-data validation:
   - Verifica positividad de todos los campos criticos
   - Handles NaN in oyster mushroom water use (valor por defecto 1500 L/kg)
   - Checks consistency between r_proteina y Proteina_pct_DM
   - Aborta con mensaje claro si hay errores criticos

4. [MEDIA] Objective-function weights parameterized:
   - WEIGHT_EARTH_DEPENDENCY = 1000.0 (configurable)
   - WEIGHT_WATER = 0.001 (configurable)

5. [MEDIA] Additional metrics in results:
   - Average area used (m2)
   - Average alternative biomass (kg/dia)

6. [LOW] Additional charts:
   - Total production evolution for K=[1,5,7,11]
   - Net-water sensitivity vs. recycling rate
""")


Grafico guardado en diversidad_tradeoff_v3.png

RESUMEN DE CORRECCIONES APLICADAS EN v3

1. [ALTA] Proteinas alternativas ahora aportan calorias:
   - Grillo: 4500 kcal/kg
   - Tenebrio: 5500 kcal/kg
   - Micoproteina: 1100 kcal/kg
   - Hongo ostra: 330 kcal/kg
   - Carne cultivada: 2500 kcal/kg
   Fuente: FAO, USDA, Oonincx et al. 2015, Finke 2002

2. [ALTA] Seccion 6b duplicada renombrada:
   - 6a: tasas diferenciadas por candidato (preservada)
   - 6b: balance hidrico completo (transpiracion vs. embebida)
   - 6c: tasas globales [90,93,95,98]% (antes era 6b duplicada)
   - 6d: comparacion balance completo vs. global (NUEVO)

3. [MEDIA] Validacion de datos de entrada:
   - Verifica positividad de todos los campos criticos
   - Maneja NaN en agua de hongo ostra (valor por defecto 1500 L/kg)
   - Verifica consistencia entre r_proteina y Proteina_pct_DM
   - Aborta con mensaje claro si hay errores criticos

4. [MEDIA] Pesos de la funcion objetivo parametrizados:
   - WEIGHT_EARTH_DEPEND

# 🧠 OR interpretation of the model

## Objective function

The model primarily minimizes dependence on resources supplied from Earth and uses water as a secondary criterion:

\[
\min \quad
w_E \cdot \text{EarthDependency}
+
w_W \cdot \text{WaterUse}
\]

where the weights are explicit parameters rather than hidden constants.

## Decisions

\[
x_{i,t} \ge 0
\]

represents the daily production of candidate \(i\) in week \(t\).

\[
y_i \in \{0,1\}
\]

indicates whether the candidate is part of the system.

## Main constraints

### Nutrition

\[
ProteinSupply_t + \Delta Protein_t \ge DemandProtein
\]

\[
CaloriesSupply_t + \Delta Calories_t \ge DemandCalories
\]

### Earth dependency

The nutritional deficit supplied from Earth is limited by a capacity that decreases over the course of the mission.

### Resources

- Maximum area
- Maximum daily water
- Maximum biomass from alternative protein sources

### Maturity

A candidate cannot produce before completing its growth cycle.

### Diversity

\[
\sum_i y_i \ge K
\]

This turns diversity into a strategic decision and makes it possible to study its **opportunity cost**.

---

## 🔬 OR insight

The interesting question is not simply:

> “Which crop produces the most?”

but:

> **“What production portfolio simultaneously satisfies multiple constraints, and what is the operational price of requiring resilience/diversity?”**

That change in perspective is precisely what turns a space-related problem into a **quantitative decision-making case**.


# 📊 How to interpret the results

When running the notebook, pay particular attention to:

### 1. `Dependencia_Tierra`
Lower values indicate greater system self-sufficiency.

### 2. `Agua_total_prom_L_dia`
Allows comparison of water efficiency across different diversity levels.

### 3. `Area_usada_prom_m2`
Measures utilization of cultivation capacity.

### 4. `Biomasa_alt_prom_kg_dia`
Shows how much the solution relies on alternative protein sources.

### 5. `K`
Shows the trade-off between **efficiency** and **resilience/diversity**.

A solution with more species may be strategically preferable even if it has a marginal resource cost. In a critical system, this difference may be more important than minimizing a single KPI.


# ⚠️ Assumptions that should be treated as hypotheses

For professional presentation of this case, it is important to distinguish between **data** and **modeling assumptions**.

The code itself identifies, among others, the following as estimates:

- calories from alternative proteins;
- water recycling rate by candidate;
- estimated water use for oyster mushroom when the data is missing;
- certain nutritional parameters.

Therefore, the results are **model results under the introduced assumptions**, not experimental predictions.

This is a strength of an OR project: a good optimizer does not only deliver a solution; it also identifies **which assumptions could change the decision**.


# 🌎 Recommended extensions

This model can evolve into a more powerful research project:

1. **Robust Optimization:** uncertainty in crop yields.
2. **Stochastic Programming:** failures, crop losses, and nutritional variability.
3. **Multi-objective Optimization:** water + mass shipped from Earth + diversity + crew time.
4. **Scenario Analysis:** crop failures or energy constraints.
5. **Monte Carlo:** probabilistic sensitivity analysis.
6. **Dynamic Optimization:** production decisions over the course of the mission.
7. **Digital Twin:** integration of operational data.
8. **Reinforcement Learning:** adaptive policy under system changes.
9. **Supply Chain Design:** incorporate inventories, resupply, and Earth–Mars lead times.

This turns the notebook into the starting point for a **Decision Science framework for autonomous systems and extreme supply chains**.
